# 08 — Paper Tables

**Project:** Causal ML for Climate Policy Effectiveness  
**Purpose:** Generate LaTeX-formatted tables for the Nexus manuscript  
**Output:** LaTeX (.tex) and CSV tables in `outputs/tables/`

---

In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(PROCESSED_DIR / "analysis_panel.csv")
print(f"Panel loaded: {panel.shape}")

Panel loaded: (7630, 77)


---
## Table 1: Summary Statistics

In [2]:
# Select key variables for summary table
summary_vars = []
var_labels = {}

# Outcome
for v, l in [("co2", "CO₂ Emissions (Mt)"), ("log_co2", "Log CO₂ Emissions"),
             ("co2_per_capita", "CO₂ per Capita (t)")]:
    if v in panel.columns:
        summary_vars.append(v)
        var_labels[v] = l

# Treatments
for v in panel.columns:
    if v.startswith("treat_"):
        summary_vars.append(v)
        var_labels[v] = v.replace("treat_", "").replace("_", " ").title()

# Covariates
for v, l in [("gdp", "GDP"), ("population", "Population"),
             ("gdp_per_capita", "GDP per Capita"),
             ("renewables_share_energy", "Renewables Share (%)"),
             ("urban_pct", "Urban Population (%)"),
             ("regulatory_quality", "Regulatory Quality")]:
    if v in panel.columns:
        summary_vars.append(v)
        var_labels[v] = l

summary = panel[summary_vars].describe().T
summary.index = [var_labels.get(v, v) for v in summary.index]
summary = summary[["count", "mean", "std", "min", "50%", "max"]]
summary.columns = ["N", "Mean", "Std. Dev.", "Min", "Median", "Max"]

print("\nTable 1: Summary Statistics")
print("=" * 75)
print(summary.round(3).to_string())

# Save as LaTeX
latex = summary.round(3).to_latex(
    caption="Summary Statistics of Key Variables",
    label="tab:summary_stats",
    position="htbp",
)
with open(TABLE_DIR / "table1_summary_stats.tex", "w", encoding="utf-8") as f:
    f.write(latex)

summary.round(3).to_csv(TABLE_DIR / "table1_summary_stats.csv")
print("\nSuccess: Table 1 saved (LaTeX + CSV)")


Table 1: Summary Statistics
                           N          Mean     Std. Dev.           Min        Median           Max
CO₂ Emissions (Mt)    7508.0  1.375440e+02  6.752860e+02  0.000000e+00  7.172000e+00  1.228904e+04
Log CO₂ Emissions     7508.0  1.867000e+00  2.853000e+00 -6.908000e+00  1.970000e+00  9.416000e+00
CO₂ per Capita (t)    7442.0  5.023000e+00  7.795000e+00  0.000000e+00  2.745000e+00  3.647910e+02
Carbon Pricing        7630.0  2.420000e-01  4.290000e-01  0.000000e+00  0.000000e+00  1.000000e+00
Renewable Mandate     7630.0  3.320000e-01  4.710000e-01  0.000000e+00  0.000000e+00  1.000000e+00
Efficiency Standard   7630.0  3.110000e-01  4.630000e-01  0.000000e+00  0.000000e+00  1.000000e+00
Fiscal Policy         7630.0  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
T1                    7630.0  2.320000e-01  4.220000e-01  0.000000e+00  0.000000e+00  1.000000e+00
T3                    7630.0  2.110000e-01  4.080000e-01  0.000000e+00  0.000000


Success: Table 1 saved (LaTeX + CSV)


---
## Table 2: Policy Coverage

In [3]:
df_policies = pd.read_csv(PROJECT_ROOT / "data" / "interim" / "policies_cleaned.csv")

# Policy type × count × country count
policy_summary = (
    df_policies
    .groupby("policy_category")
    .agg(
        n_policies=("iso3", "count"),
        n_countries=("iso3", "nunique"),
        earliest=("year_adopted", "min"),
        latest=("year_adopted", "max"),
    )
    .reset_index()
)
policy_summary.columns = ["Policy Type", "N Policies", "N Countries", "Earliest", "Latest"]

print("\nTable 2: Climate Policy Coverage")
print("=" * 65)
print(policy_summary.to_string(index=False))

latex = policy_summary.to_latex(
    index=False,
    caption="Climate Policy Coverage by Type",
    label="tab:policy_coverage",
)
with open(TABLE_DIR / "table2_policy_coverage.tex", "w") as f:
    f.write(latex)

policy_summary.to_csv(TABLE_DIR / "table2_policy_coverage.csv", index=False)
print("Success: Table 2 saved")


Table 2: Climate Policy Coverage
        Policy Type  N Policies  N Countries  Earliest  Latest
     Carbon Pricing         910           82      1944    2025
Efficiency Standard        2664           93      1965    2025
              Other        8739          194      1927    2026
  Renewable Mandate         854          147      1991    2026
Success: Table 2 saved


---
## Table 3: Main Results (Method Comparison)

In [4]:
try:
    comparison = pd.read_csv(TABLE_DIR / "method_comparison.csv")
    
    print("\nTable 3: Main Results — Average Treatment Effects")
    print("=" * 80)
    print(comparison.to_string(index=False))
    
    latex = comparison.to_latex(
        index=False,
        caption="Average Treatment Effects: OLS, DML, and Causal Forest Estimates",
        label="tab:main_results",
    )
    with open(TABLE_DIR / "table3_main_results.tex", "w") as f:
        f.write(latex)
    print("Success: Table 3 saved")
except FileNotFoundError:
    print("Warning: Run Notebook 05 first")


Table 3: Main Results — Average Treatment Effects
       Method                 Treatment  Estimate  Std. Error            95% CI  p-value    N Significant
          OLS      treat_carbon_pricing    0.1936      0.0196  [0.1551, 0.2321]   0.0000 2028         ***
          OLS   treat_renewable_mandate    0.1055      0.0190  [0.0684, 0.1427]   0.0000 2028         ***
          OLS treat_efficiency_standard    0.2165      0.0251  [0.1672, 0.2657]   0.0000 2028         ***
          OLS                  treat_t1    0.1627      0.0190  [0.1254, 0.1999]   0.0000 2028         ***
          OLS                  treat_t3    0.1181      0.0184  [0.0822, 0.1541]   0.0000 2028         ***
          OLS                  treat_t5    0.0556      0.0178  [0.0208, 0.0904]   0.0017 2028         ***
DML (xgboost)      treat_carbon_pricing    0.0899      0.0154  [0.0598, 0.1200]   0.0000 2028         ***
DML (xgboost)   treat_renewable_mandate    0.0203      0.0143 [-0.0077, 0.0482]   0.1549 2028        

---
## Table 4: Robustness Summary

In [5]:
try:
    alt_specs = pd.read_csv(TABLE_DIR / "alternative_specifications.csv")
    
    print("\nTable 4: Robustness — Alternative Specifications")
    print("=" * 70)
    print(alt_specs.to_string(index=False))
    
    latex = alt_specs.to_latex(
        index=False,
        caption="Robustness: Alternative Covariate Specifications",
        label="tab:robustness",
    )
    with open(TABLE_DIR / "table4_robustness.tex", "w") as f:
        f.write(latex)
    print("Success: Table 4 saved")
except FileNotFoundError:
    print("Warning: Run Notebook 06 first")


Table 4: Robustness — Alternative Specifications
specification  n_covariates  estimate  ci_lower  ci_upper  std_error  n_obs
      Minimal             2 -0.219579 -0.255611 -0.183547   0.018384   6845
     Baseline             4 -0.193926 -0.227814 -0.160039   0.017290   6655
     Extended             6  0.104574  0.068651  0.140498   0.018329   2028
Success: Table 4 saved


In [6]:
print("\n" + "=" * 60)
print("Success: All paper tables generated!")
print("=" * 60)

# List all generated tables
table_files = sorted(TABLE_DIR.glob("*.tex"))
csv_files = sorted(TABLE_DIR.glob("*.csv"))

print(f"\nLaTeX tables ({len(table_files)}):")
for f in table_files:
    print(f"  - {f.name}")

print(f"\nCSV tables ({len(csv_files)}):")
for f in csv_files:
    print(f"  - {f.name}")

print("\nSuccess: Pipeline complete! Ready for manuscript writing.")


Success: All paper tables generated!

LaTeX tables (4):
  - table1_summary_stats.tex
  - table2_policy_coverage.tex
  - table3_main_results.tex
  - table4_robustness.tex

CSV tables (7):
  - alternative_specifications.csv
  - method_comparison.csv
  - sector_estimates.csv
  - summary_statistics.csv
  - table1_summary_stats.csv
  - table2_policy_coverage.csv
  - temporal_dynamics.csv

Success: Pipeline complete! Ready for manuscript writing.
